In [1]:
import numpy as np
import pyreadr
import pickle
import pandas as pd
from pathlib import Path
from tqdm import tqdm
import geopandas as gpd
from scipy.spatial.distance import pdist, squareform
from scipy.sparse import csr_matrix
from scipy.sparse.csgraph import connected_components

np.random.seed(1)

# ------------------------------------------------------------
# CONFIG
# ------------------------------------------------------------

BASE_DIR = Path(r"D:\77\Research\temp\snow")
PERIOD = 52
N_REP = 1000

no_nbs = np.array([
    57,170,236,269,343,685,946,947,989,
    1037,1084,1090,1109,1118,1127,1176,1203
]) - 1

# ------------------------------------------------------------
# LOAD DATA + 2PC FILTER (SAME AS SPATIAL NB)
# ------------------------------------------------------------

snow = pyreadr.read_r(BASE_DIR / "snow_cleaned_full.Rda")
snow = list(snow.values())[0]

coords_full = snow.iloc[:, :2].to_numpy()
y_full = snow.iloc[:, 2:].to_numpy()

coords_tmp = np.delete(coords_full, no_nbs, axis=0)
y_tmp = np.delete(y_full, no_nbs, axis=0)

# build adjacency
gdf = gpd.GeoDataFrame(
    geometry=gpd.points_from_xy(coords_tmp[:,0], coords_tmp[:,1]),
    crs="EPSG:4326"
).to_crs("+proj=aeqd +lat_0=90 +lon_0=-100")

xy = np.vstack([gdf.geometry.x, gdf.geometry.y]).T / 1e6
D = squareform(pdist(xy))

W_full = (D <= 0.22).astype(int)
np.fill_diagonal(W_full, 0)
W_full = csr_matrix(W_full)

n_comp, labels = connected_components(W_full, directed=False)
sizes = np.bincount(labels)
order = np.argsort(sizes)[::-1]

keep = np.sort(np.concatenate([
    np.where(labels == order[0])[0],
    np.where(labels == order[1])[0]
]))

coords = coords_tmp[keep]
y = y_tmp[keep]
W = W_full[keep][:, keep]

S, T = y.shape
print("Using S =", S)

# ------------------------------------------------------------
# SPATIO-TEMPORAL SNOW-CONDITIONED STATISTIC
# ------------------------------------------------------------

def st_neighbor_stat(y_mat):

    total_neighbors = 0
    total_snow = 0

    for t in range(1, T-1):

        yt = y_mat[:, t]
        snow_mask = yt == 1
        n_snow = snow_mask.sum()

        if n_snow == 0:
            continue

        total_snow += n_snow

        # spatial neighbors
        spatial_counts = W @ yt.astype(int)
        spatial_part = spatial_counts[snow_mask].sum()

        # temporal neighbors
        temporal_part = (
            y_mat[:, t-1][snow_mask].sum() +
            y_mat[:, t+1][snow_mask].sum()
        )

        total_neighbors += spatial_part + temporal_part

    return total_neighbors / total_snow


T_obs = st_neighbor_stat(y)
print("Observed ST neighbor:", T_obs)

# ------------------------------------------------------------
# TIME COVARIATES (SAME AS SPATIAL NB)
# ------------------------------------------------------------

t_raw = np.arange(1, T+1)
t_trend = (t_raw - t_raw.mean()) / t_raw.std()

cov4 = np.column_stack([
    np.ones(T),
    np.cos(2*np.pi*t_raw/PERIOD),
    np.sin(2*np.pi*t_raw/PERIOD),
    t_trend
])

cov8 = np.column_stack([
    np.ones(T), np.ones(T),
    np.cos(2*np.pi*t_raw/PERIOD), np.cos(2*np.pi*t_raw/PERIOD),
    np.sin(2*np.pi*t_raw/PERIOD), np.sin(2*np.pi*t_raw/PERIOD),
    t_trend, t_trend
])

week = (t_raw - 1) % 52

# ------------------------------------------------------------
# SPATIAL COVARIATES (SAME AS SPATIAL NB)
# ------------------------------------------------------------

lat = coords[:,1]
lat = (lat - lat.mean()) / lat.std()

elev_df = pd.read_csv(BASE_DIR / "curr_elev.csv")
elev_all = elev_df.iloc[:,3].to_numpy()
elev = elev_all[keep]
elev = (elev - elev.mean()) / elev.std()

snow_temp = pyreadr.read_r(BASE_DIR / "snow_temp_full.Rda")
snow_temp = list(snow_temp.values())[0].reset_index(drop=True)
temp_all = snow_temp.drop(index=no_nbs).iloc[:,2:].to_numpy()
temp = temp_all[keep]
temp = (temp - temp.mean()) / temp.std()

# ------------------------------------------------------------
# LOAD POSTERIORS (SAME FILES AS SPATIAL NB)
# ------------------------------------------------------------

iid01 = np.load(BASE_DIR / "ind01.npz")["all_theta"]
iid10 = np.load(BASE_DIR / "ind10.npz")["all_theta"]

bym01 = np.load(BASE_DIR / "bym01.npz")["all_theta"]
bym10 = np.load(BASE_DIR / "bym10.npz")["all_theta"]

bym01_cov = np.load(BASE_DIR / "bym01_cov.npz")["all_theta"]
bym10_cov = np.load(BASE_DIR / "bym10_cov.npz")["all_theta"]

with open(BASE_DIR / "bym01_weekly.pkl","rb") as f:
    weekly01 = pickle.load(f)
with open(BASE_DIR / "bym10_weekly.pkl","rb") as f:
    weekly10 = pickle.load(f)

eta01_week = weekly01["all_eta"]
tau01_week = weekly01["all_tau"]
eta10_week = weekly10["all_eta"]
tau10_week = weekly10["all_tau"]

with open(BASE_DIR / "bym01_cov_weekly.pkl","rb") as f:
    wf01 = pickle.load(f)
with open(BASE_DIR / "bym10_cov_weekly.pkl","rb") as f:
    wf10 = pickle.load(f)

eta01_wf = wf01["all_eta"]
tau01_wf = wf01["all_tau"]
eta10_wf = wf10["all_eta"]
tau10_wf = wf10["all_tau"]

M = iid01.shape[1]

def sigmoid(z):
    z = np.clip(z,-30,30)
    return 1/(1+np.exp(-z))

# ------------------------------------------------------------
# GENERIC ONE-STEP-AHEAD SIMULATOR
# ------------------------------------------------------------

def simulate(get_eta):

    draws = np.random.choice(M, N_REP, replace=False)
    stats = np.zeros(N_REP)

    for j, m in enumerate(tqdm(draws)):

        y_rep = np.zeros((S,T), dtype=int)
        y_rep[:,0] = y[:,0]

        for t in range(1,T):

            eta01, eta10 = get_eta(m,t)

            p01 = sigmoid(eta01)
            p10 = sigmoid(eta10)

            prev = y[:,t-1]   # one-step-ahead (observed)

            prob = np.where(prev==0, p01, 1-p10)

            y_rep[:,t] = np.random.binomial(1, prob)

        stats[j] = st_neighbor_stat(y_rep)

    return stats

# ------------------------------------------------------------
# ETA DEFINITIONS (SAME STRUCTURE AS SPATIAL NB)
# ------------------------------------------------------------

def eta_iid(m,t):
    e1 = sum(cov4[t,k]*iid01[k*S:(k+1)*S,m] for k in range(4))
    e2 = sum(cov4[t,k]*iid10[k*S:(k+1)*S,m] for k in range(4))
    return e1,e2

def eta_bym(m,t):
    e1 = sum(cov8[t,k]*bym01[k*S:(k+1)*S,m] for k in range(8))
    e2 = sum(cov8[t,k]*bym10[k*S:(k+1)*S,m] for k in range(8))
    return e1,e2

def eta_week(m,t):
    w = week[t]
    e1 = np.zeros(S)
    e2 = np.zeros(S)
    for k in range(8):
        beta1 = eta01_week[k*S:(k+1)*S,m]
        beta2 = eta10_week[k*S:(k+1)*S,m]
        e1 += cov8[t,k]*beta1*tau01_week[k*52+w,m]
        e2 += cov8[t,k]*beta2*tau10_week[k*52+w,m]
    return e1,e2

def eta_cov(m,t):
    e1,e2 = eta_bym(m,t)
    gamma1 = bym01_cov[8*S:8*S+3,m]
    gamma2 = bym10_cov[8*S:8*S+3,m]
    fac = np.column_stack([
        t_trend[t]*lat,
        t_trend[t]*elev,
        t_trend[t]*temp[:,t]
    ])
    return e1+fac@gamma1, e2+fac@gamma2

def eta_week_cov(m,t):
    w = week[t]
    e1 = np.zeros(S)
    e2 = np.zeros(S)
    for k in range(8):
        beta1 = eta01_wf[k*S:(k+1)*S,m]
        beta2 = eta10_wf[k*S:(k+1)*S,m]
        e1 += cov8[t,k]*beta1*tau01_wf[k*52+w,m]
        e2 += cov8[t,k]*beta2*tau10_wf[k*52+w,m]
    gamma1 = eta01_wf[8*S:8*S+3,m]
    gamma2 = eta10_wf[8*S:8*S+3,m]
    fac = np.column_stack([
        t_trend[t]*lat,
        t_trend[t]*elev,
        t_trend[t]*temp[:,t]
    ])
    return e1+fac@gamma1, e2+fac@gamma2

# ------------------------------------------------------------
# RUN
# ------------------------------------------------------------

print("\nIID")
I_iid = simulate(eta_iid)

print("\nBYM shared")
I_bym = simulate(eta_bym)

print("\nBYM weekly")
I_week = simulate(eta_week)

print("\nBYM + cov")
I_cov = simulate(eta_cov)

print("\nBYM weekly + cov")
I_week_cov = simulate(eta_week_cov)

# ------------------------------------------------------------
# SUMMARY
# ------------------------------------------------------------

def summarize(name, x):
    print("\n", name)
    print("Mean:", x.mean())
    print("SD:", x.std())
    print("P(stat >= obs):", np.mean(x >= T_obs))

summarize("IID", I_iid)
summarize("BYM shared", I_bym)
summarize("BYM weekly", I_week)
summarize("BYM+cov", I_cov)
summarize("BYM weekly+cov", I_week_cov)

Using S = 1557
Observed ST neighbor: 5.058381036721609

IID


100%|██████████| 1000/1000 [15:55<00:00,  1.05it/s]



BYM shared


100%|██████████| 1000/1000 [25:30<00:00,  1.53s/it]



BYM weekly


...


BYM + cov


100%|██████████| 1000/1000 [28:59<00:00,  1.74s/it]



BYM weekly + cov


100%|██████████| 1000/1000 [23:19<00:00,  1.40s/it]


 IID
Mean: 4.7084260648827065
SD: 0.001284069629158002
P(stat >= obs): 0.0

 BYM shared
Mean: 4.710664483658468
SD: 0.0013042993295436837
P(stat >= obs): 0.0

 BYM weekly
Mean: 4.704611834037167
SD: 0.0013328481331692308
P(stat >= obs): 0.0

 BYM+cov
Mean: 4.706839134803376
SD: 0.0014865863087671605
P(stat >= obs): 0.0

 BYM weekly+cov
Mean: 4.705834744696077
SD: 0.0012633828921211018
P(stat >= obs): 0.0
